# Оценка качества QA-генерации с помощью метрик (Трек A)

**Задача:** оценить и сравнить качество ответов на вопросы (Question Answering) у трёх
бесплатных LLM через OpenRouter.

**Почему Трек A (QA).** QA даёт самый прямой сигнал о *фактической точности* модели:
есть эталонный короткий ответ, и метрики (F1, Exact Match) измеряют именно его
воспроизведение, а не правдоподобность текста. Это удобно для интерпретации ошибок и
для решения «какую модель брать под фактологические задачи».

**Модели (OpenRouter, free):**
- `meta-llama/llama-3.3-70b-instruct:free`
- `google/gemini-2.0-flash-exp:free`
- `qwen/qwen-2.5-72b-instruct:free`

**Датасет:** [`kuznetsoffandrey/sberquad`](https://huggingface.co/datasets/kuznetsoffandrey/sberquad) —
русская версия SQuAD (контекст + вопрос + эталонный ответ). Берём срез из 80 примеров
из валидационного сплита и фиксируем его в `qa_dataset.jsonl` для воспроизводимости.

**Метрики:** Exact Match, token-level F1, BLEU, Semantic Similarity, время генерации,
длина ответа (реализация — в `metrics.py`).

> Запускать рекомендуется в Google Colab: локально на этой машине инференс
> sentence-transformers/transformers падает с segfault.


## 0. Установка зависимостей

In [ ]:
!pip -q install datasets sentence-transformers sacrebleu openai pandas matplotlib scipy python-dotenv

## 1. Импорты и ключи

Ключ `OPENROUTER_API_KEY` берём из `.env` (локально) или из Colab Secrets / переменной окружения.

In [ ]:
import os, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd

# .env (локально) — в Colab можно прописать os.environ напрямую или через userdata
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# В Colab: from google.colab import userdata; os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Не найден OPENROUTER_API_KEY (положи в .env или Colab Secrets)"

import metrics  # наш модуль metrics.py (должен лежать рядом с ноутбуком)
print("metrics.py загружен:", [m for m in dir(metrics) if not m.startswith('_')][:8])

## 2. Датасет: срез SberQuAD

Загружаем валидационный сплит, берём 80 примеров с непустым ответом и сохраняем срез.

In [ ]:
from datasets import load_dataset

N = 80
SEED = 42
ds = load_dataset("kuznetsoffandrey/sberquad", split="validation")  # org sberbank-ai переименован

rows = []
for ex in ds:
    ans = ex["answers"]["text"]
    if ans and ans[0].strip():
        rows.append({
            "id": ex.get("id", len(rows)),
            "context": ex["context"],
            "question": ex["question"],
            "answer": ans[0].strip(),
        })

import random
random.Random(SEED).shuffle(rows)
data = rows[:N]

with open("qa_dataset.jsonl", "w", encoding="utf-8") as f:
    for r in data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Сохранено {len(data)} примеров в qa_dataset.jsonl")
pd.DataFrame(data)[["question", "answer"]].head()

## 3. Клиент OpenRouter и промпт-форматы

Используем OpenAI-совместимый клиент с base_url OpenRouter. Встроен ретрай на 429
(free-модели часто упираются в rate limit).

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

MODELS = [
    "meta-llama/llama-3.3-70b-instruct:free",
    "google/gemini-2.0-flash-exp:free",
    "qwen/qwen-2.5-72b-instruct:free",
]

def call_model(model, system, user, temperature=0.0, max_retries=5):
    """Запрос к модели с ретраем на 429/5xx. Возвращает (text, latency_sec)."""
    delay = 4.0
    for attempt in range(max_retries):
        t0 = time.time()
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                temperature=temperature,
                max_tokens=128,
            )
            latency = time.time() - t0
            return (resp.choices[0].message.content or "").strip(), latency
        except Exception as e:
            msg = str(e)
            if ("429" in msg or "rate" in msg.lower() or "5" == msg[:1]) and attempt < max_retries - 1:
                time.sleep(delay); delay *= 1.8; continue
            return f"[ERROR] {msg}", time.time() - t0
    return "[ERROR] retries exhausted", 0.0

### Промпт-форматы

Четыре формата для экспериментов Части 3:
- **short** — минимальная инструкция (тест влияния длины промпта);
- **detailed** — подробная инструкция с требованием краткости (A/B vs short);
- **fewshot** — detailed + 2 примера (zero-shot vs few-shot);

Ответ просим максимально короткий — так честнее считается Exact Match.

In [ ]:
SHORT_SYS = "Отвечай на вопрос по тексту."

DETAILED_SYS = (
    "Ты — точная QA-система. Ответь на вопрос, используя ТОЛЬКО приведённый контекст. "
    "Дай максимально короткий ответ (одно слово или фраза), без пояснений, без точки в конце. "
    "Если ответа в тексте нет — напиши 'нет ответа'."
)

FEWSHOT_EXAMPLES = (
    "Пример 1:\nКонтекст: Москва — столица России.\nВопрос: Какая столица России?\nОтвет: Москва\n\n"
    "Пример 2:\nКонтекст: Роман написан в 1869 году Львом Толстым.\nВопрос: В каком году написан роман?\nОтвет: 1869\n\n"
)

def build_user(context, question, fewshot=False):
    prefix = FEWSHOT_EXAMPLES if fewshot else ""
    return f"{prefix}Контекст: {context}\nВопрос: {question}\nОтвет:"

PROMPT_VARIANTS = {
    "short":    dict(system=SHORT_SYS,    fewshot=False),
    "detailed": dict(system=DETAILED_SYS, fewshot=False),
    "fewshot":  dict(system=DETAILED_SYS, fewshot=True),
}

## 4. Прогон моделей

`run_experiment` гоняет один (модель × формат × температура) по всему срезу и возвращает
DataFrame с предсказаниями, временем и длиной ответа. Результаты кэшируем на диск,
чтобы не платить токенами при повторных запусках анализа.

In [ ]:
def run_experiment(model, variant, temperature=0.0, data=data):
    cfg = PROMPT_VARIANTS[variant]
    out = []
    for r in data:
        user = build_user(r["context"], r["question"], fewshot=cfg["fewshot"])
        pred, lat = call_model(model, cfg["system"], user, temperature=temperature)
        out.append({
            "id": r["id"], "model": model, "variant": variant, "temperature": temperature,
            "question": r["question"], "reference": r["answer"],
            "prediction": pred, "latency": lat, "pred_len": len(pred),
        })
    return pd.DataFrame(out)

def cached_run(model, variant, temperature=0.0):
    safe = model.replace("/", "_").replace(":", "_")
    path = Path(f"cache_{safe}_{variant}_t{temperature}.csv")
    if path.exists():
        return pd.read_csv(path)
    df = run_experiment(model, variant, temperature)
    df.to_csv(path, index=False)
    return df

### 4.1 Базовый прогон: все 3 модели, формат `detailed`, t=0.0

In [ ]:
frames = []
for m in MODELS:
    print("→", m)
    frames.append(cached_run(m, "detailed", temperature=0.0))
base = pd.concat(frames, ignore_index=True)
base[["model", "question", "reference", "prediction", "latency"]].head(6)

## 5. Подсчёт метрик

Семантическую близость считаем батчем одной мультиязычной моделью
`paraphrase-multilingual-MiniLM-L12-v2`. Остальные метрики — построчно из `metrics.py`.

In [ ]:
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def add_metrics(df):
    df = df.copy()
    df["em"]  = [metrics.exact_match(p, r) for p, r in zip(df.prediction, df.reference)]
    df["f1"]  = [metrics.token_f1(p, r)   for p, r in zip(df.prediction, df.reference)]
    df["bleu"]= [metrics.compute_bleu(p, r) for p, r in zip(df.prediction, df.reference)]
    df["sem"] = metrics.semantic_similarity(df.prediction.tolist(), df.reference.tolist(), st_model)
    return df

base_m = add_metrics(base)

def summarize(df, by=("model",)):
    by = list(by)
    g = df.groupby(by).agg(
        EM=("em", "mean"), F1=("f1", "mean"), BLEU=("bleu", "mean"),
        SemSim=("sem", "mean"), Latency_s=("latency", "mean"), PredLen=("pred_len", "mean"),
    ).round(3)
    return g.sort_values("F1", ascending=False)

summary = summarize(base_m)
summary

### 5.1 Визуализация сравнения моделей

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
summary[["EM", "F1", "SemSim"]].plot.bar(ax=ax[0], rot=15)
ax[0].set_title("Качество по моделям"); ax[0].set_ylim(0, 1)
summary[["Latency_s"]].plot.bar(ax=ax[1], rot=15, color="indianred", legend=False)
ax[1].set_title("Среднее время генерации, с")
plt.tight_layout(); plt.show()

## 6. Эксперименты (Часть 3)

### 6.1 Zero-shot vs Few-shot и влияние формата/длины промпта

Гоняем три формата (`short`, `detailed`, `fewshot`) для каждой модели и сравниваем.

In [ ]:
rows = []
for m in MODELS:
    for v in ["short", "detailed", "fewshot"]:
        rows.append(add_metrics(cached_run(m, v, temperature=0.0)))
prompt_exp = pd.concat(rows, ignore_index=True)

prompt_summary = summarize(prompt_exp, by=["model", "variant"])
prompt_summary

In [ ]:
# Эффект формата промпта на F1 (усреднённо по моделям)
pivot = prompt_exp.groupby("variant").agg(F1=("f1","mean"), EM=("em","mean"), SemSim=("sem","mean")).round(3)
pivot = pivot.reindex(["short", "detailed", "fewshot"])
print(pivot)
pivot.plot.bar(rot=0, figsize=(7,4), title="Влияние формата промпта (среднее по моделям)")
plt.ylim(0,1); plt.show()

### 6.2 Влияние температуры генерации

Для лучшей по F1 модели гоняем формат `detailed` при t = 0.0 / 0.3 / 0.7.

In [ ]:
best_model = summary.index[0]
print("Лучшая по F1 модель:", best_model)

temp_rows = []
for t in [0.0, 0.3, 0.7]:
    temp_rows.append(add_metrics(cached_run(best_model, "detailed", temperature=t)))
temp_exp = pd.concat(temp_rows, ignore_index=True)

temp_summary = temp_exp.groupby("temperature").agg(
    F1=("f1","mean"), EM=("em","mean"), SemSim=("sem","mean"), PredLen=("pred_len","mean")
).round(3)
temp_summary

### 6.3 Анализ типичных ошибок

Смотрим худшие случаи (низкий F1, но иногда высокая семантика — признак того, что
модель права по смыслу, но не совпала по форме) и лучшие случаи.

In [ ]:
b = base_m.copy()
worst = b.sort_values("f1").head(8)[["model","question","reference","prediction","f1","sem"]]
best  = b[b.f1 > 0.99].head(6)[["model","question","reference","prediction","f1","sem"]]
print("=== ХУДШИЕ (низкий F1) ===")
display(worst)
print("=== ЛУЧШИЕ (F1≈1) ===")
display(best)

# Сколько случаев "семантически верно, но EM=0" — расхождение формы и смысла
mism = b[(b["em"] == 0) & (b["sem"] > 0.75)]  # b["sem"], т.к. b.sem — это метод DataFrame
print(f"\nСемантически близко, но Exact Match=0: {len(mism)} из {len(b)} "
      f"({100*len(mism)/len(b):.0f}%) — типичная проблема EM на русском.")

## 7. Статистическая значимость (бонус)

Bootstrap-доверительные интервалы F1 для каждой модели и парный bootstrap-тест:
значимо ли отличается лучшая модель от остальных.

In [ ]:
print("95% CI для F1 (bootstrap):")
ci = {}
for m in MODELS:
    vals = base_m[base_m.model == m].f1.values
    mean, lo, hi = metrics.bootstrap_ci(vals)
    ci[m] = (mean, lo, hi)
    print(f"  {m:45s}  F1={mean:.3f}  CI=[{lo:.3f}, {hi:.3f}]")

print("\nПарный bootstrap-тест (H0: F1 равны), относительно лучшей модели:")
best_vals = base_m[base_m.model == best_model].sort_values("id").f1.values
for m in MODELS:
    if m == best_model:
        continue
    other = base_m[base_m.model == m].sort_values("id").f1.values
    p = metrics.paired_bootstrap_pvalue(best_vals, other)
    verdict = "значимо" if p < 0.05 else "НЕ значимо"
    print(f"  {best_model.split('/')[-1]} vs {m.split('/')[-1]:35s}  p={p:.4f}  ({verdict})")

## 8. Итоговая таблица и сохранение результатов

In [ ]:
all_results = pd.concat([prompt_exp, temp_exp], ignore_index=True).drop_duplicates(
    subset=["id","model","variant","temperature"])
all_results.to_csv("results.csv", index=False)
summary.to_csv("summary.csv")
print("Сохранено: results.csv, summary.csv")
summary

## 9. Выводы

*(заполняется по факту прогона — краткие тезисы)*

- **Лучшая модель по F1/SemSim:** см. таблицу `summary` — рекомендация под фактологический QA.
- **Формат промпта:** `detailed` обычно > `short`; few-shot даёт прирост EM за счёт
  стабилизации формата короткого ответа.
- **Температура:** для QA оптимум около t=0.0 (рост t повышает многословность и снижает EM).
- **Информативность метрик:** F1 и SemSim — самые информативные; **Exact Match занижен**
  на русском (перефразирование, согласование падежей); BLEU слабо работает на коротких
  ответах. Подробный разбор — в `report.md`.
